1. 安装环境

In [ ]:
!pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
!pip install ultralytics
# pip install ultralytics 时默认安装cpu版的pytorch，需要额外装cuda版的pytorch
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
# 由于ultralytics模块需要调用opencv，但该容器没有opengl，因此要把opencv模块改成安装opencv-python-headless
!pip uninstall -y opencv-python
!pip install opencv-python-headless

2. 准备数据集
   - 数据集存放在挂载的对象存储中: /mnt/data/traffic

In [3]:
import os
dataset_path = r'/mnt/data/traffic'
print(os.listdir(dataset_path))

['images', 'labels']


数据集的配置文件：/mnt/data/mobole.yaml

In [4]:
import yaml
dataset_yaml = r'/mnt/data/mobile.yaml'
with open(dataset_yaml, 'r', encoding='utf-8') as file:
    data = yaml.safe_load(file)
    print(data)

{'path': '/mnt/data/traffic', 'train': ['images/train'], 'val': ['images/val'], 'names': {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'tricycle'}}


3. 模型训练

In [ ]:
import torch
from ultralytics import YOLO
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('/mnt/data/yolo26n.pt')
model.train(data=dataset_yaml, 
            epochs=3, # 迭代轮次
            patience=5, # 提前停止条件
            batch=64, # batchsize
            imgsz=640, # 输入尺寸
            device=device, # 训练设备
            project='/mnt/data/run/train', # 保存路径
            name='mobile',  # 保存训练结果和权重的文件名称
            optimizer='AdamW'  # 优化器 SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp
            )

Ultralytics 8.4.39 🚀 Python-3.13.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA A10, 22503MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/data/mobile.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/mnt/data/yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mobile-9, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=5

3.1 数据增强
可使用参数：
model.train(
    ...
    hsv_h,
    hsv_s,
    hsv_v,
    degrees, # 支持0-180°的旋转
    scale, # 缩放
    translate, # 水平/垂直方向移动
    shear, # 仿射变换
    perspective, # 透视变换
    flipud, # 垂直翻转
    fliplr, # 水平翻转
    mosaic, # 把4张照片裁剪、缩放合成1张图片，yolo默认的数据增强方法
    mixup，# 2张图片线性叠加
    cutmix, # 挖取一张图片的区域，粘贴到另外一张图片的相对位置
)

3.2 策略增强
可使用参数：
model.train(
    ...
    optimizer，  # 优化器 SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp
    dropout, 
    warmup_epochs,
    lr0, # 学习率，
    lrf, # lr = lrf*lr0
    momentum, # optimizer=Adam的参数
    dfl, # focal loss
)


4. 模型验证

In [5]:
from ultralytics import YOLO
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('/mnt/data/run/train/mobile-9/weights/best.pt')
model.val(data=dataset_yaml, 
          imgsz=640, 
          batch=32, 
          conf=0.5, 
          iou=0.5, 
          device=device,
          project='/mnt/data/run/val',
          name='mobile')

Ultralytics 8.4.39 🚀 Python-3.13.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA A10, 22503MiB)
YOLO26n summary (fused): 122 layers, 2,375,811 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 5.4±5.0 MB/s, size: 139.6 KB)
val: Scanning /mnt/data/traffic/labels/val.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 49.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 3.0s/it 47.4s0.1s
                   all        500       4177      0.603      0.208      0.182     0.0937
                person        442       1954      0.827      0.451      0.424      0.247
               bicycle         61         85        0.8     0.0471      0.043     0.0267
                   car        270        912      0.707       0.18      0.145     0.0733
            motorcycle        475       1225      0.678      0.362      0.296      0.121
              tricycle          1          1 

5. 模型推理

In [6]:
from ultralytics import YOLO
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('/mnt/data/run/train/mobile-7/weights/best.pt')
model.predict(source='/mnt/data/ultralytics/ultralytics/assets', 
              imgsz=640, 
              conf=0.5, 
              iou=0.5,
              batch=1,
              device=device,
              project='/mnt/data/run/val',
              name='mobile')


image 1/2 /mnt/data/ultralytics/ultralytics/assets/bus.jpg: 640x480 1 person, 59.1ms
image 2/2 /mnt/data/ultralytics/ultralytics/assets/zidane.jpg: 384x640 (no detections), 128.5ms
Speed: 1.4ms preprocess, 93.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)


6. 模型导出

In [ ]:
from ultralytics import YOLO

model = YOLO("path/to/best.pt")  
model.export(format="onnx", # 支持'onnx', 'torchscript', 'engine' 和其它
             imagsz = 640, # 输入尺寸，也支持[640, 480]格式
             half = True,  # 支持半浮点精度
             dynamic = True  # 若开启，bacth, height, width均为动态输入
             )